In [ ]:
import struct
import numpy as np
import math
from enum import Enum
import sys
sys.path.append('/Users/abhinavarora/Desktop/Machine Learning')
import custom_math
import random

#Parsing the binary format
with open("/Users/abhinavarora/Desktop/Machine Learning/Neural Network/MNIST handwritten /train-images-idx3-ubyte/train-images-idx3-ubyte", "rb") as f:
    magic, num, rows, cols = struct.unpack(">IIII", f.read(16))
    images = np.frombuffer(f.read(), dtype=np.uint8)
    images = images.reshape(num, rows * cols)

#Loading in the labels
with open("/Users/abhinavarora/Desktop/Machine Learning/Neural Network/MNIST handwritten /train-labels-idx1-ubyte/train-labels-idx1-ubyte", "rb") as f:
    magic, num = struct.unpack(">II", f.read(8))
    labels = np.frombuffer(f.read(), dtype=np.uint8)


#This is to one-hot encode input labels whenever softmax regression is used.
#The class number simply is the number of classes
def vectorize_labels(input_labels, class_number):
    #The one-hot encoded arrays are being stored in a matrix where each column corresponds to the number of classes
    #And the row corresponds to which class-type. This is consistent with the dimensions of the output of the final layer
    res = [[0] * len(input_labels) for _ in range(class_number)]
    for i in range(len(input_labels)):
        #one-hot encoding
        res[input_labels[i]][i] = 1
    
    return res

vectorized_labels = vectorize_labels(labels, 10)
print(len(vectorized_labels), len(vectorized_labels[0]))

In [ ]:
img_dimension = rows * cols
#Each image flattened into a vector and put into a matrix
image_input_matrix = [[0] * len(images) for _ in range(img_dimension)]

for column in range(len(images)):
    for row in range(len(images[0])):
        #Normalizing to fit every value between 0 and 1 to solve vanishing gradients
        image_input_matrix[row][column] = images[column][row]/255


num_examples = len(image_input_matrix[0])
train_size = int(0.8 * num_examples)

# Shuffle indices
indices = list(range(num_examples))
random.shuffle(indices)

train_indices = indices[:train_size]
test_indices = indices[train_size:]

# Split inputs — selecting columns corresponding to each split
x_train = [[image_input_matrix[row][col] for col in train_indices] for row in range(len(image_input_matrix))]
x_test  = [[image_input_matrix[row][col] for col in test_indices]  for row in range(len(image_input_matrix))]

# Split labels the same way   
y_train = [[vectorized_labels[row][col] for col in train_indices] for row in range(len(vectorized_labels))]
y_test  = [[vectorized_labels[row][col] for col in test_indices]  for row in range(len(vectorized_labels))]

In [ ]:
class ActivationFunctions:
    #Required to know the number of neurons since activation functions will be vector operations 
    def __init__(self):
        pass

    def sigmoid(self, input_val):
        exponent = math.exp(-input_val)
        return (1/(1+exponent))
    
    def ReLU(self, input_val):
        if input_val > 0:
            return input_val
        return 0

    
class FunctionType(Enum):
    RELU = 1
    SIGMOID = 2

#This is a layer of neurons with specific number of neurons that is a hyper parameter
#As well as the number of inputs this layer will take
class Layer:
    def __init__(self, neuron_num, input_size, function_type: FunctionType):
        self.function_type = function_type
        self.neuron_num = neuron_num
        self.input_size = input_size
        #Generates a random number in the normal distribution with mean 0 and std 0.01. This results in values between
        #-0.03 and 0.03. This is required since we want to break symmetry in each layer of a neural network and not 
        #make them identical
        self.parameters = [[random.gauss(0, 0.01) for _ in range(input_size)] for _ in range(neuron_num)]
        self.bias = [[0] for _ in range(neuron_num)]
        #Linear part and result of the activation being stored directly as a property of the layer itself
        self.z = None
        self.a = None
        #Required for Adam
        self.first_moment_weight = [[0] * input_size for _ in range(neuron_num)]
        self.second_moment_weight = [[0] * input_size for _ in range(neuron_num)]
        self.first_moment_bias = [[0] for _ in range(neuron_num)]
        self.second_moment_bias = [[0] for _ in range(neuron_num)]
    
    def execute_function(self, input_val):
        #Initialising the class of the different activation functions
        functions = ActivationFunctions()
        if self.function_type == FunctionType.RELU:
            return functions.ReLU(input_val)
        if self.function_type == FunctionType.SIGMOID:
            return functions.sigmoid(input_val)
    
    #Returns the omega * x + b
    #Bias automatically gets broadcasted in this function
    def linearize(self, parameters, input, bias):
        parameter_input_product = custom_math.matrix_with_matrix_multiplication(parameters, input)
        #Broadcast
        #This is the number of rows of the broadcasted bias matrix
        dimension = len(parameter_input_product[0])
        broadcasted_matrix = [[0] * dimension for _ in range(len(parameter_input_product))]
        for row in range(len(broadcasted_matrix)):
            for col in range(len(broadcasted_matrix[0])):
                broadcasted_matrix[row][col] = bias[row][0]
        return custom_math.matrix_addition_and_sub(parameter_input_product, broadcasted_matrix, "add")

    #The softmax function is only for the last layer's output, by default it will be set to 0. However, the underlying
    #hypothesis function will not change
    def hypothesis(self, linear, softmax=False):
        linear_copy = [[0] * len(linear[0]) for _ in range(len(linear))]
        for row in range(len(linear)):
            for col in range(len(linear[0])):
                linear_copy[row][col] = self.execute_function(linear[row][col])
        
        if not softmax:
            return linear_copy
        else:
            for col in range(len(linear[0])):
                #Calculating the total exponential sum for each ouptut
                exponential_sum = 0
                for row in range(len(linear)):
                    exponential_sum += math.exp(linear_copy[row][col])
                
                #Applying the softmax formula
                for row in range(len(linear)):
                    linear_copy[row][col] = math.exp(linear_copy[row][col])/exponential_sum
        
        return linear_copy
    
class LossFunctions:
    def __init__(self):
        pass

    #This is the cross entropy function required
    #y_hat is the final prediction vector and y is the actual label vector
    def cross_entropy_loss(self, y_hat, y):
        regresularised_predictions = self.regularise(y_hat)
        for i in range(len(regresularised_predictions)):
            regresularised_predictions[i] = math.log(regresularised_predictions[i]) * y[i]
        
        res = sum(i for i in regresularised_predictions)
        return -res
    
    #This is a function that will add an epsilon value (extremely small) to prevent from obtaining the log (0) in any calculation
    def regularise(self, vector, epsilon = 1e-5):
        vector_copy = [0] * len(vector)
        for i in range(len(vector)):
            vector_copy[i] = vector[i] + epsilon
        
        return vector_copy


In [ ]:
#Support for Adam and Standard Gradient Descent for now
class Optimizer:
    def __init__(self, learning_rate):
        self.learning_rate = learning_rate
    
    def update(self, layers:list[Layer], dw, db):
        pass


class Adam(Optimizer):
    beta1 = 0.9
    beta2 = 0.999

    def __init__(self, learning_rate):
        super().__init__(learning_rate)
        self.t = 0
    
    def update(self, layers:list[Layer], dw, db):
        self.t += 1
        bias_corr1 = 1 - self.beta1 ** self.t
        bias_corr2 = 1 - self.beta2 ** self.t

        for i in range(len(layers)):
            #Updating first moment for weights: m = beta1*m + (1-beta1)*dw
            term1_weight_moment_one = custom_math.scalar_multiply_matrix(layers[i].first_moment_weight, self.beta1)
            term2_weight_moment_one = custom_math.scalar_multiply_matrix(dw[i], 1 - self.beta1)
            layers[i].first_moment_weight = custom_math.matrix_addition_and_sub(term1_weight_moment_one, term2_weight_moment_one, "add")

            #Updating second moment for weights: v = beta2*v + (1-beta2)*dw^2
            term1_weight_moment_two = custom_math.scalar_multiply_matrix(layers[i].second_moment_weight, self.beta2)
            derivatives_weight_squared = custom_math.element_wise_multiplication(dw[i], dw[i])
            term2_weight_moment_two = custom_math.scalar_multiply_matrix(derivatives_weight_squared, 1 - self.beta2)
            layers[i].second_moment_weight = custom_math.matrix_addition_and_sub(term1_weight_moment_two, term2_weight_moment_two, "add")

            #Updating first moment for bias: m = beta1*m + (1-beta1)*db
            term1_bias_moment_one = custom_math.scalar_multiply_matrix(layers[i].first_moment_bias, self.beta1)
            term2_bias_moment_one = custom_math.scalar_multiply_matrix(db[i], 1 - self.beta1)
            layers[i].first_moment_bias = custom_math.matrix_addition_and_sub(term1_bias_moment_one, term2_bias_moment_one, "add")

            #Updating second moment for bias: v = beta2*v + (1-beta2)*db^2
            term1_bias_moment_two = custom_math.scalar_multiply_matrix(layers[i].second_moment_bias, self.beta2)
            derivatives_bias_squared = custom_math.element_wise_multiplication(db[i], db[i])
            term2_bias_moment_two = custom_math.scalar_multiply_matrix(derivatives_bias_squared, 1 - self.beta2)
            layers[i].second_moment_bias = custom_math.matrix_addition_and_sub(term1_bias_moment_two, term2_bias_moment_two, "add")

            #Bias correction: m_hat = m / (1 - beta^t)
            m_hat_weight = custom_math.scalar_multiply_matrix(layers[i].first_moment_weight, 1 / bias_corr1)
            v_hat_weight = custom_math.scalar_multiply_matrix(layers[i].second_moment_weight, 1 / bias_corr2)
            m_hat_bias = custom_math.scalar_multiply_matrix(layers[i].first_moment_bias, 1 / bias_corr1)
            v_hat_bias = custom_math.scalar_multiply_matrix(layers[i].second_moment_bias, 1 / bias_corr2)

            #Updating weight: params -= lr * m_hat / (sqrt(v_hat) + eps)
            root_v_weight = custom_math.element_wise_roots(v_hat_weight, 2)
            epsilon_matrix_w = [[1e-8] * len(v_hat_weight[0]) for _ in range(len(v_hat_weight))]
            denominator_weight = custom_math.matrix_addition_and_sub(root_v_weight, epsilon_matrix_w, "add")
            step_weight = custom_math.scalar_multiply_matrix(custom_math.element_wise_division_two_matrices(m_hat_weight, denominator_weight), self.learning_rate)
            layers[i].parameters = custom_math.matrix_addition_and_sub(layers[i].parameters, step_weight, "sub")

            #Updating bias
            root_v_bias = custom_math.element_wise_roots(v_hat_bias, 2)
            epsilon_matrix_b = [[1e-8] * len(v_hat_bias[0]) for _ in range(len(v_hat_bias))]
            denominator_bias = custom_math.matrix_addition_and_sub(root_v_bias, epsilon_matrix_b, "add")
            step_bias = custom_math.scalar_multiply_matrix(custom_math.element_wise_division_two_matrices(m_hat_bias, denominator_bias), self.learning_rate)
            layers[i].bias = custom_math.matrix_addition_and_sub(layers[i].bias, step_bias, "sub")

class SGD(Optimizer):
    def __init__(self, learning_rate):
        super().__init__(learning_rate)
    def update(self, layers:list[Layer], dw, db):
        for i in range(len(layers)):
            with_learning_rate_weight = custom_math.scalar_multiply_matrix(dw[i], self.learning_rate)
            with_learning_rate_bias = custom_math.scalar_multiply_matrix(db[i], self.learning_rate)
            layers[i].parameters = custom_math.matrix_addition_and_sub(layers[i].parameters, with_learning_rate_weight, "sub")
            layers[i].bias = custom_math.matrix_addition_and_sub(layers[i].bias, with_learning_rate_bias, "sub")


In [ ]:
class Network:

    def __init__(self, layer_num, neurons_in_layers, initial_input, function_type:FunctionType, optimizer:Optimizer):
        #This initialises the number of layers
        self.number_of_layers = layer_num
        #This is a list that specifies the number of neurons in each layer 
        self.neurons_in_layers = neurons_in_layers
        self.initial_input = initial_input
        #This is the array that stores the actual layer objects
        self.layers:list[Layer] = []
        self.optimizer = optimizer
        #Initialising the layers
        for i in range(self.number_of_layers):
            #This is specifically for the first layer. This is because the input_size is the dimension of the vector of each training example
            if i == 0:
                layer = Layer(self.neurons_in_layers[i], len(self.initial_input), function_type)
            #For the other layers, the input size the number of neurons of the previous layer since each neuron outputs a single number
            else:
                layer = Layer(self.neurons_in_layers[i], self.neurons_in_layers[i-1], function_type)
            
            self.layers.append(layer)

    #This is the feedforward function. This will be a recursive function.
    #The layer_index specifies which layer in self.layers and input specifies the input for each layer
    def feedforward(self, layer_index, input):
        #Base case
        if layer_index >= len(self.layers):
            #Technically this is the final output now
            return
        layer:Layer = self.layers[layer_index]
        linear_res = layer.linearize(layer.parameters, input, layer.bias)
        layer.z = linear_res
        if layer_index == len(self.layers) - 1:
            output = layer.hypothesis(linear_res, True)
        else:
            output = layer.hypothesis(linear_res)
        #Useful for caching results
        layer.a = output
        layer.z = linear_res
        #The input of the next layer becomes the output of the current layer
        return self.feedforward(layer_index + 1, output)
    
    #THe total loss function is the sum of the loss functions across each layer.
    #The output is the output of the final layer
    def total_loss(self, output, loss_type, input_labels):
        total_loss = 0
        #The output has dimension (number of neurons in last layer, training examples) but this makes it hard to iterate over each column
        #Therefore, the transpose allows us to iterate row by row
        #Same logic for input_labels since they were one-hot encoded to be in the same dimension as the output
        output_transpose = custom_math.transpose_matrix(output)
        input_labels_transpose = custom_math.transpose_matrix(input_labels)
        for row in range(len(output_transpose)):
            total_loss += loss_type(output_transpose[row], input_labels_transpose[row])
        
        total_loss /= len(input_labels_transpose)
        return total_loss
    
    #The following backprop functions are hardcoded for binary cross entropy. It is not practical to have such hardcoded backprop functions 
    #and so an autograd engine will be implemented later on. There is a possibility that the previous_layer is none and so the input is just the initial input
    def last_layer_backprop(self, labels, final_layer:Layer, prev_layer: Layer = None):
        if prev_layer != None:
            prev_activation_transpose = custom_math.transpose_matrix(prev_layer.a)
        else:
            prev_activation_transpose = custom_math.transpose_matrix(self.initial_input)
        
        term_one = custom_math.matrix_addition_and_sub(labels, final_layer.a, "sub")
        final_prod = custom_math.matrix_with_matrix_multiplication(term_one, prev_activation_transpose)
        #The total loss is the average of losses across each training example
        res = custom_math.scalar_multiply_matrix(final_prod, -1/len(labels[0]))
        #This is the product that will be backpropagated. According to calculations, the product backpropagated to the previous
        #layer (W[l-1]) is the same as dJ/dW[l] without A[l-1]T.
        product_two = custom_math.scalar_multiply_matrix(term_one, -1/len(labels[0]))
        return res, product_two
    
    #This is the general pattern for the backprop for previous layers. 
    def previous_layer_backprop(self, current_layer:Layer, next_layer:Layer, previous_product, activation:FunctionType, previous_layer:Layer = None):
        next_layer_parameter_transpose = custom_math.transpose_matrix(next_layer.parameters)
        multiplied_term_one = custom_math.matrix_with_matrix_multiplication(next_layer_parameter_transpose, previous_product)
        product_one = custom_math.element_wise_multiplication(multiplied_term_one, current_layer.a)
        if activation == FunctionType.SIGMOID:
            matrix_of_ones = [[1] * len(current_layer.a[0]) for _ in range(len(current_layer.a))]
            term = custom_math.matrix_addition_and_sub(matrix_of_ones, current_layer.a, "sub")
            product_two = custom_math.element_wise_multiplication(product_one, term)
        elif activation == FunctionType.RELU:
            product_two = custom_math.element_wise_multiplication(multiplied_term_one, custom_math.ReLU_derivative(current_layer.z))
        previous_layer_activation_transpose = custom_math.transpose_matrix(previous_layer.a) if previous_layer != None else custom_math.transpose_matrix(self.initial_input) 
        res = custom_math.matrix_with_matrix_multiplication(product_two, previous_layer_activation_transpose)
        #res is dJ/dW[l] of the current layer whereas product_two is the actual product that will be backpropagated.
        #product_two is also dJ/dB[l]
        return res, product_two
    
    #The training loop is as follows:
    #1)Feedforward and store produced activation, parameters and bias in the layer
    #2)Backprop and update each parameter
    #3)Repeat until parameter convergence
    def train_loop(self, epochs, train_labels):
        
        for _ in range(epochs):
            self.feedforward(0, self.initial_input)
            loss_fn = LossFunctions()
            current_loss = self.total_loss(self.layers[-1].a, loss_fn.cross_entropy_loss, train_labels)
            print(f"Epoch {_}, Loss: {current_loss}")

            weight_results = [0] * len(self.layers)
            bias_results = [0] * len(self.layers)
            #The backprop_prod is the same as dJ/dB[l]!!!
            last_layer_res, backprop_prod = self.last_layer_backprop(train_labels, self.layers[-1], self.layers[-2]) if len(self.layers) > 1 else self.last_layer_backprop(train_labels, self.layers[0])
            weight_results[-1] = last_layer_res
            bias_results[-1] = [[sum(backprop_prod[row])] for row in range(len(backprop_prod))]
            for i in range(len(self.layers) - 2, -1, -1):
                if i > 0:
                    res, backprop_prod = self.previous_layer_backprop(self.layers[i], self.layers[i+1], backprop_prod, self.layers[i].function_type, self.layers[i-1])
                else:
                    res, backprop_prod = self.previous_layer_backprop(self.layers[i], self.layers[i+1], backprop_prod, self.layers[i].function_type)
                bias_results[i] = [[sum(backprop_prod[row])] for row in range(len(backprop_prod))]
                weight_results[i] = res
            
            #Update rule
            self.optimizer.update(self.layers, weight_results, bias_results)



In [ ]:
#Runs a forward pass on x_test and returns the percentage of correctly classified examples.
#The predicted class is the argmax of each output column, same for the actual label.
def test_accuracy(self, x_test, y_test):
    self.feedforward(0, x_test)
    output = self.layers[-1].a
    correct = 0
    for col in range(len(output[0])):
        pred = max(range(len(output)), key=lambda row: output[row][col])
        actual = max(range(len(y_test)), key=lambda row: y_test[row][col])
        if pred == actual:
            correct += 1
    return (correct / len(output[0])) * 100

In [ ]:
#Now finally training the Neural Network !!!!
adam = Adam(0.01)
MNIST_Network = Network(3, [128, 64, 10], x_train, FunctionType.RELU, adam)
MNIST_Network.train_loop(2000, y_train)